In [1]:
!pip install groq
!pip install gradio

from groq import Groq

client = Groq(api_key="Your API Key here")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.9 MB/s eta 0:00:00


In [19]:
import requests
from datetime import datetime
import xml.etree.ElementTree as ET
import gradio as gr
try:
    from IPython.display import display, Markdown
    notebook = True
except ImportError:
    notebook = False


#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                 llama usage function                    #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/

def llama_generate(prompt, max_tokens=1500):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "Tu es un assistant scientifique expert."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,  # 0.3 for Similar summary for articles 1 and 2
        max_tokens=max_tokens
    )
    return response.choices[0].message.content



#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                 Tool for arXiv shearch                   #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/

def arxiv_search(query, max_results):
    url = f"http://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results={max_results}"
    response = requests.get(url)
    root = ET.fromstring(response.content)
    ns = {'atom': 'http://www.w3.org/2005/Atom'}
    results = []
    # metadata retrieval
    for entry in root.findall('atom:entry', ns):
        title = entry.find('atom:title', ns).text.strip()
        summary = entry.find('atom:summary', ns).text.strip()
        link = entry.find('atom:id', ns).text.strip()
        authors = [author.find('atom:name', ns).text.strip() for author in entry.findall('atom:author', ns)]
        published = entry.find('atom:published', ns).text[:10]
        results.append({
    "title": title,
    "authors": authors,
    "summary": summary,
    "url": link,
    "date": published
})
    return results



#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                        Article info                      #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/

def enrich_metadata(selected_articles, all_articles):
    enriched = []

    for sel in selected_articles:
        match = next(
            (a for a in all_articles if a["title"] == sel["title"]),
            None
        )

        if match:
            sel["authors"] = match.get("authors", [])
            sel["date"] = match.get("date", "Unknown")
            sel["url"] = match.get("url", "")

        enriched.append(sel)

    return enriched


#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                        Search Agent                      #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#


def agent_recherche(task, arxiv_count=5):
    print(f"\n=== Agent Recherche : {task} ===\nDate : {datetime.now().strftime('%Y-%m-%d')}\n")

    # Tool using
    arxiv_results = arxiv_search(task, max_results=arxiv_count)
    for a in arxiv_results:
        a["source"] = "arXiv"

    return arxiv_results



#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                      Selection Agent                     #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#

def agent_selection(all_articles, task):

    articles_text = "\n".join([
      f"""{i+1}. {a['title']}
      Authors: {', '.join(a.get('authors', []))}
      Date: {a.get('date', 'Unknown')}
      Summary: {a['summary'][:200]}
      URL: {a['url']}
      """
    for i, a in enumerate(all_articles)
    ])

    prompt = f"""
You are a scientific research expert.

Here are the results from a search on: "{task}".

Articles found:
{articles_text}

Your task:
Select the **two most relevant and important articles** for the given research topic.

Requirements:
- The two articles must be **different**.
- Consider the **title, summary, and source** when determining relevance.
- Focus on **importance, clarity, and relevance** to the topic.
- Respond strictly in the following **JSON format**:

[
  {{
    "title": "Article title",
    "summary": "Concise summary",
    "authors": ["Author 1", "Author 2"],
    "date": "YYYY-MM-DD",
    "source": "arXiv",
    "url": "Link"
  }},
  {{
    "title": "...",
    "summary": "...",
    "authors": [...],
    "date": "...",
    "source": "...",
    "url": "..."
  }}
]

- Select only **two articles**.
- Respond **only with valid JSON**. Do **not** include explanations, notes, or any extra text.
"""


    try:
        selection = llama_generate(prompt)
    except TypeError:

        selection = llama_generate(prompt)
    return selection



#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                        Editor Agent                      #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#

def agent_redacteur_par_article(selected_articles):
    all_resumes = []

    for i, article in enumerate(selected_articles, start=1):
        prompt = f"""
You are a highly skilled scientific writing assistant.

Your task:
Create a **very detailed, structured, and comprehensive summary** of the following scientific article.
The summary should be precise, understandable, and suitable for a researcher familiar with the field. Include the main ideas, key findings, methodology if relevant, and any important context.

Article information:
- Title: {article['title']}
- Source: {article['source']}
- Original Abstract: {article['summary']}
- URL: {article['url']}

Instructions:
- Expand the summary significantly beyond the original abstract.
- Explain ALL important concepts in detail
- Make it informative and coherent, suitable for someone doing scientific research.
- Do not omit important information from the original abstract.
- Write in fluent English.
- Include context, background, and explanations
- Make the summary understandable but deep

STRICT FORMAT (MUST FOLLOW EXACTLY):

### 1. Overview
Brief explanation of the article and its purpose.

### 2. Methodology
Describe the methods, models, or approach used.

### 3. Key Findings
List the main results and contributions.

### 4. Implications
Explain why this work is important.

Rules:
- Use ALL sections (even if brief)
- Use the EXACT section titles
- No extra sections
- No introduction text before sections
- No conclusion outside sections
- Write in clear academic English

"""
        resume = llama_generate(prompt)

        all_resumes.append({
              "title": article['title'],
              "resume": resume,
              "url": article['url'],
              "source": article['source'],
              "authors": article.get("authors", []),
              "date": article.get("date", "Unknown")
        })

    return all_resumes



#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                        Judge Agent                       #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#

def agent_judge(resumes,selected_articles ):

    improved_resumes = []

    for r in resumes:

        articles_text = "\n".join([
            f"{i+1}. {a['title']} ({a.get('source', 'Unknown')}): {a['summary'][:200]}" # 200 for less context yes, but faster processing time (15s less)
            for i, a in enumerate(selected_articles)
        ])

        prompt = f"""
You are a highly experienced scientific review agent.

Original summary for an article:
{r['resume']}

Context from related articles (ArXiv):
{articles_text}

Your task:
- Carefully evaluate the original summary in terms of **accuracy, completeness, clarity, and relevance** to the research topic.
- Identify any missing key points, unclear statements, or gaps in explanation.
- Produce a **fully improved version** of the summary that is more complete, precise, and clear.
- Structure the improved summary in coherent paragraphs, highlighting important points, methodology, and key findings if relevant.
- Enhance readability and ensure it reflects the scientific context accurately.
- Use fluent, formal English suitable for a researcher.
- Respond **only with the improved summary text**. Do **not** include the original summary, comments, or any additional information.
"""


        try:
            improved = llama_generate(prompt)
        except TypeError:
            improved = llama_generate(prompt)

        improved_resumes.append({
            "title": r["title"],
            "improved_resume": improved,
            "authors": r.get("authors", []),
            "date": r.get("date", "Unknown"),
            "url": r.get("url", "")
        })

    return improved_resumes



#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                      Format Agent                     #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#


def agent_format(arxiv_articles,improved_resumes):

    formatted = ""
      # Format of all article find
    formatted += "# All Articles Found (arXiv)\n\n"
    for i, a in enumerate(arxiv_articles, start=1):
        authors = ", ".join(a.get("authors", ["Unknown"]))
        date = a.get("date", "Unknown")
        url = a.get("url", "#")
        formatted += f"{i}. **{a['title']}**\n"
        formatted += f"   - Authors: {authors}\n"
        formatted += f"   - Date: {date}\n"
        formatted += f"   - Link: [{url}]({url})\n\n"

    formatted += "# Selected Research Articles\n\n"
        # info retrival
    for i, r in enumerate(improved_resumes, start=1):
        title = r.get("title", "N/A")
        authors = ", ".join(r.get("authors", ["Unknown"]))
        date = r.get("date", "Unknown")
        url = r.get("url", "#")
        summary = r.get("improved_resume", "")

        formatted += f"## {i}. {title}\n\n"

        # Format Métadonnées
        formatted += f"- **Authors:** {authors}\n"
        formatted += f"- **Publication Date:** {date}\n"
        formatted += f"- **Link:** [{url}]({url})\n\n"

        # format summary
        formatted += f"### Summary\n\n{summary}\n\n"

        formatted += "---\n\n"

    return formatted



#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                          Pipeline                        #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#

def research_pipeline(task, output_format="markdown"):


    #   Call Research agent
    arxiv_articles = agent_recherche(task, arxiv_count=10)

    # Call selection agent
    selection_json = agent_selection(arxiv_articles, task)

    #  convertion llama slection output (JSON) to python list
    import json
    try:
        selected_articles = json.loads(selection_json)
    except:
        print(" Error parsing JSON → fallback arXiv")
        selected_articles = [
            {
                "title": a["title"],
                "summary": a["summary"],
                "source": "arXiv",
                "url": a["url"],
                "authors": a.get("authors", ["Unknown"]),
                "date": a.get("date", "Unknown")
            }
            for a in arxiv_articles[:2]
        ]

    selected_articles = enrich_metadata(selected_articles, arxiv_articles)

    # article storage
    article1 = selected_articles[0]
    article2 = selected_articles[1]

    # Call editor agent
    resumes = agent_redacteur_par_article([article1, article2])

    # Call judge agent
    improved_resumes = agent_judge(resumes, arxiv_articles)

    # Call format agent
    formatted_output = agent_format(arxiv_articles,improved_resumes)

    status = ""

    return formatted_output, status



#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#/
#                          Gradio UI                       #
#/#/#/#/#/#/#/#/#/#/#/#/#//#/#/#/#/#/#//#/#//#/#/#/#/#/#/#/#

with gr.Blocks(css="""
.title-box {
    background-color: black;
    border: 3px solid orange;
    color: white;
    text-align: center;
    padding: 15px;
    border-radius: 10px;
    font-size: 28px;
    font-weight: bold;
    margin-bottom: 20px;
}

.subtitle {
    text-align: center;
    color: white;
    font-size: 16px;
    margin-bottom: 10px;
}

.warning {
    text-align: center;
    color: orange;
    font-size: 15px;
    margin-bottom: 20px;
    font-weight: bold;
}

/* bouton orange */
button {
    background-color: orange !important;
    color: white !important;
    font-weight: bold;
}

/* status box styling */
.status-box {
    text-align: center;
    color: orange;
    font-weight: bold;
    font-size: 14px;
}
""") as UI:

    # Title
    gr.Markdown('<div class="title-box">Scientific Paper Research Agent on ArXiv</div>')

    # sentance
    gr.Markdown("""
<div class="subtitle">
This AI agent searches scientific paper on ArXiv, selects the most relevant articles,
and generates a clear and improved summary using multiple intelligent agents.
</div>
""")

    # time warning
    gr.Markdown("""
<div class="warning">
⚠️ The research process may take around 40 seconds. Please be patient.
</div>
""")

    # raw organisation
    with gr.Row():
        query = gr.Textbox(
            label="Research Topic",
            placeholder="Enter a research topic here...",
            elem_id="query-box",
            scale=4
        )
        status_box = gr.Label(
            value="",
            elem_classes="status-box",
            scale=1
        )

    #  Markdown output
    output_box = gr.Markdown(label="Research Results")

    # Bouton
    run_btn = gr.Button("Run Research")

    # parameters bouton
    run_btn.click(
        fn=research_pipeline,
        inputs=query,
        outputs=[output_box, status_box]
    )
    # enter to start like bouton
    query.submit(
        fn=research_pipeline,
        inputs=query,
        outputs=[output_box, status_box]
    )

UI.launch()

/tmp/ipykernel_440/3522276188.py:384: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css="""


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40e1de30f624c8da23.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
